# AgroDataLab EnviroPro - Analisis reproducible

Notebook de apoyo para la entrega. Usa el dataset principal `data/processed/enviropro_completo_2024_2026.csv` y genera el informe y las 12 visualizaciones pedidas en `reports/`.

## Objetivos

- Cargar y normalizar lecturas EnviroPro.
- Revisar calidad de datos: duplicados, huecos temporales, nulos y valores incoherentes.
- Calcular indicadores de humedad, temperatura, bateria y panel solar.
- Generar visualizaciones defendibles para la memoria.
- Mantener IFAPA-RIA fuera de Django, como ampliacion opcional.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "processed" / "enviropro_completo_2024_2026.csv"
REPORT_PATH = ROOT / "reports" / "analisis_enviropro.md"
FIGURES_DIR = ROOT / "reports" / "figures"

DATA_PATH

In [ ]:
df = pd.read_csv(DATA_PATH)
df["fecha_hora"] = pd.to_datetime(df["fecha_hora"], errors="coerce")
df = df.sort_values("fecha_hora").reset_index(drop=True)

humidity_fields = [f"hum_s{i}_media" for i in range(1, 9)]
temp_mean_fields = [f"temp_s{i}_media" for i in range(1, 9)]
temp_min_fields = [f"temp_s{i}_min" for i in range(1, 9)]
temp_max_fields = [f"temp_s{i}_max" for i in range(1, 9)]

df["humedad_media"] = df[humidity_fields].mean(axis=1)
df["temperatura_media"] = df[temp_mean_fields].mean(axis=1)
df["temperatura_minima"] = df[temp_min_fields].min(axis=1)
df["temperatura_maxima"] = df[temp_max_fields].max(axis=1)
df["bateria_v"] = df["bateria_mv"] / 1000
df["panel_solar_v"] = df["panel_solar_mv"] / 1000

df[["fecha_hora", "humedad_media", "temperatura_media", "bateria_v", "panel_solar_v"]].head()

In [ ]:
quality_summary = {
    "registros": len(df),
    "primera_lectura": df["fecha_hora"].min(),
    "ultima_lectura": df["fecha_hora"].max(),
    "duplicados_exactos": int(df.duplicated().sum()),
    "fechas_duplicadas": int(df["fecha_hora"].duplicated().sum()),
    "huecos_mayores_2h": int((df["fecha_hora"].diff() > pd.Timedelta(hours=2)).sum()),
    "temperaturas_incoherentes": int(((df["temperatura_maxima"] - df["temperatura_minima"]) > 18).sum()),
    "bateria_baja": int((df["bateria_v"] < 6.2).sum()),
    "panel_solar_bajo": int((df["panel_solar_v"] < 1.0).sum()),
}
quality_summary

## Generacion del informe

La logica completa esta en `notebooks/analisis_enviropro.py`. Ejecutar la celda siguiente regenera el informe Markdown y las 12 figuras usadas por la web y la memoria.

In [ ]:
%run notebooks/analisis_enviropro.py

In [ ]:
figures = sorted(FIGURES_DIR.glob("*.png"))
[figure.name for figure in figures]

## Visualizaciones generadas

1. Humedad media temporal.
2. Temperatura media temporal.
3. Bateria temporal.
4. Panel solar temporal.
5. Humedad por sensor.
6. Temperatura promedio por sensor.
7. Evolucion semanal de humedad.
8. Evolucion semanal de temperatura.
9. Correlacion entre humedad, temperatura y energia.
10. Dispersion humedad media frente a temperatura media.
11. Alertas por dia.
12. Valores nulos o problemas por sensor.

In [ ]:
print(REPORT_PATH.read_text(encoding="utf-8")[:2500])